# Predicting Attention Deficits in Children Using EEG and Behavioral Data
**Dataset:** OpenNeuro ds005507 — Healthy Brain Network (HBN-EEG) Release 3  
**Target:** CBCL Attention T-score → Binary (Low/High) + Continuous regression  
**Kernel:** Python (attention_eeg)

---
## Pipeline Overview
```
1. Setup & Dataset Download
2. EEG Preprocessing  (filter → re-reference → ICA → epoch)
3. Feature Extraction (band power, theta/alpha ratio, entropy, coherence, behavioral)
4. Exploratory Data Analysis
5. Model Training     (Classification + Regression, 5-fold CV)
6. Evaluation & Metrics
7. Visualization      (ROC, confusion matrix, SHAP, learning curve, feature importance)
```

---
## Cell 1 — Imports & Configuration

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib

import mne
mne.set_log_level('WARNING')

from scipy.signal import welch
import antropy as ant
import shap

from sklearn.model_selection import StratifiedKFold, KFold, learning_curve, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge, ElasticNet
from sklearn.svm import SVC, SVR
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, cohen_kappa_score,
    confusion_matrix, roc_curve, auc, ConfusionMatrixDisplay,
    mean_absolute_error, r2_score
)
import xgboost as xgb

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR      = os.getcwd()
DATA_RAW      = os.path.join(BASE_DIR, 'data', 'raw')
DATA_PROC     = os.path.join(BASE_DIR, 'data', 'processed')
RESULTS_PLOTS = os.path.join(BASE_DIR, 'results', 'plots')
RESULTS_MODELS= os.path.join(BASE_DIR, 'results', 'models')
for d in [DATA_RAW, DATA_PROC, RESULTS_PLOTS, RESULTS_MODELS]:
    os.makedirs(d, exist_ok=True)

# ── Config ─────────────────────────────────────────────────────────────────
DATASET_ID       = 'ds005507'
MAX_SUBJECTS     = 50          # increase once you confirm pipeline works
EPOCH_DURATION   = 2.0         # seconds
L_FREQ, H_FREQ   = 1.0, 45.0
NOTCH_FREQ       = 60.0
REJECT_THRESH    = 150e-6      # 150 µV
ATTENTION_CUTOFF = 65.0        # CBCL clinical threshold
N_FOLDS          = 5
RANDOM_STATE     = 42
N_CHANNELS       = 19

BANDS = {'delta':(1,4), 'theta':(4,8), 'alpha':(8,13), 'beta':(13,30), 'gamma':(30,45)}
META_COLS = ['subject_id','attention','attention_high',
             'p_factor','internalizing','externalizing','age','sex']

plt.rcParams.update({'font.family':'DejaVu Sans',
                     'axes.spines.top':False, 'axes.spines.right':False})
print('✓ Imports and config ready')
print(f'  Working dir : {BASE_DIR}')
print(f'  Max subjects: {MAX_SUBJECTS}')

---
## Cell 2 — Download Dataset (ds005507)

In [ ]:
import openneuro
from tqdm.notebook import tqdm

participants_file = os.path.join(DATA_RAW, 'participants.tsv')

# ── Step 1: Download metadata if missing ──────────────────────────────────
if not os.path.exists(participants_file):
    print(f'Downloading metadata for {DATASET_ID}...')
    openneuro.download(
        dataset=DATASET_ID,
        target_dir=DATA_RAW,
        include=['participants.tsv', 'participants.json', 'dataset_description.json'],
    )
    print('✓ Metadata downloaded')
else:
    print('✓ participants.tsv already present')

# ── Step 2: Download EEG folders for each subject (skip if already present) ─
pf = pd.read_csv(participants_file, sep='\t')
pf = pf.dropna(subset=['attention'])
subjects = pf['participant_id'].tolist()[:MAX_SUBJECTS]   # keeps 'sub-NDAR...' prefix

print(f'\nChecking / downloading EEG for {len(subjects)} subjects...')
already, downloaded, failed = 0, 0, 0

for subj in tqdm(subjects):
    eeg_dir = os.path.join(DATA_RAW, subj, 'eeg')
    if os.path.isdir(eeg_dir) and any(
        f.endswith(('.set', '.edf', '.bdf', '.fif')) for f in os.listdir(eeg_dir)
    ):
        already += 1
        continue   # EEG files already on disk

    try:
        openneuro.download(
            dataset=DATASET_ID,
            target_dir=DATA_RAW,
            include=[f'{subj}/eeg/'],
        )
        downloaded += 1
    except RuntimeError as e:
        print(f'  WARN {subj}: {e}')
        failed += 1

print(f'\n✓ Done — already present: {already} | newly downloaded: {downloaded} | failed: {failed}')

# ── Show dataset overview ─────────────────────────────────────────────────
participants_df = pd.read_csv(participants_file, sep='\t').dropna(subset=['attention'])
print(f'\nSubjects with attention score : {len(participants_df)}')
print(f'Age range : {participants_df["age"].min():.1f} – {participants_df["age"].max():.1f} years')
print(f'Sex       : {participants_df["sex"].value_counts().to_dict()}')
print(f'\nAttention score stats:')
print(participants_df['attention'].describe().round(2))
participants_df.head()

---
## Cell 3 — EEG Preprocessing
Pipeline: **Bandpass filter → Notch → Average re-reference → ICA artifact removal → Epoch → Reject bad epochs**

In [ ]:
from mne_bids import BIDSPath, read_raw_bids

def preprocess_subject(subject_id):
    """Full EEG preprocessing for one subject. Returns clean epochs or None."""
    try:
        bids_path = BIDSPath(
            subject=subject_id, task='RestingState',
            datatype='eeg', root=DATA_RAW
        )
        raw = read_raw_bids(bids_path, verbose=False)
        raw.load_data()

        # Keep only EEG channels — HBN-EEG has NO dedicated EOG electrodes
        raw.pick_types(eeg=True, stim=False, exclude='bads')

        # 1. Bandpass + notch filter
        raw.filter(l_freq=L_FREQ, h_freq=H_FREQ, fir_design='firwin', verbose=False)
        raw.notch_filter(freqs=NOTCH_FREQ, verbose=False)

        # 2. Average re-reference
        raw.set_eeg_reference('average', projection=True)
        raw.apply_proj()

        # 3. ICA — muscle artifact removal only (NO EOG detection)
        n_eeg = len(mne.pick_types(raw.info, eeg=True))
        n_comp = min(20, n_eeg - 1)
        ica = mne.preprocessing.ICA(n_components=n_comp, random_state=RANDOM_STATE,
                                     max_iter='auto')
        ica.fit(raw)
        try:
            muscle_idx, _ = ica.find_bads_muscle(raw)
            ica.exclude = list(set(muscle_idx))
        except Exception:
            pass
        raw = ica.apply(raw)

        # 4. Fixed-length epochs (2 s)
        epochs = mne.make_fixed_length_epochs(raw, duration=EPOCH_DURATION,
                                               preload=True, verbose=False)
        epochs.drop_bad(reject={'eeg': REJECT_THRESH})

        if len(epochs) < 5:
            return None

        # Save
        out = os.path.join(DATA_PROC, f'sub-{subject_id}_epochs.fif')
        epochs.save(out, overwrite=True)
        return epochs

    except Exception as e:
        print(f'  SKIP {subject_id}: {e}')
        return None

# ── Run preprocessing — only subjects with downloaded eeg folders ─────────
pf = pd.read_csv(participants_file, sep='\t').dropna(subset=['attention'])
all_subjects = pf['participant_id'].str.replace('sub-', '').tolist()[:MAX_SUBJECTS]

subjects = [s for s in all_subjects
            if os.path.isdir(os.path.join(DATA_RAW, f'sub-{s}', 'eeg'))]

print(f'Found {len(subjects)} subjects with downloaded EEG data\n')

success, failed = 0, 0
for subj in tqdm(subjects):
    out_path = os.path.join(DATA_PROC, f'sub-{subj}_epochs.fif')
    if os.path.exists(out_path):
        print(f'  already preprocessed: {subj}')
        success += 1
        continue
    result = preprocess_subject(subj)
    if result is not None:
        success += 1
        print(f'  OK {subj} — {len(result)} clean epochs')
    else:
        failed += 1

print(f'\nPreprocessing done  |  Success: {success}  |  Failed: {failed}')

---
## Cell 4 — Visualize One Subject's EEG (Quality Check)

In [ ]:
%matplotlib inline

epoch_files = [f for f in os.listdir(DATA_PROC) if f.endswith('_epochs.fif')]
if epoch_files:
    sample_path = os.path.join(DATA_PROC, epoch_files[0])
    sample_epochs = mne.read_epochs(sample_path, verbose=False, preload=True)
    subj_name = epoch_files[0].replace('_epochs.fif', '')

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # PSD
    sample_epochs.compute_psd(fmin=1, fmax=45, picks='eeg').plot(axes=axes[0], show=False)
    axes[0].set_title(f'Power Spectral Density — {subj_name}', fontsize=12, fontweight='bold')

    # Epoch image — use standalone function instead of method to avoid axes issue
    data_ch0 = sample_epochs.get_data()[:, 0, :]
    axes[1].imshow(data_ch0, aspect='auto', cmap='RdBu_r',
                   extent=[0, data_ch0.shape[1]/sample_epochs.info['sfreq'], len(data_ch0), 0])
    axes[1].set_xlabel('Time (s)')
    axes[1].set_ylabel('Epoch')
    axes[1].set_title('Epoch Image (Ch 0)', fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_PLOTS, '0_eeg_quality_check.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\nSubject: {subj_name}')
    print(f'Clean epochs: {len(sample_epochs)}')
    print(f'Channels: {len(sample_epochs.ch_names)}')
    print(f'Sampling rate: {sample_epochs.info["sfreq"]} Hz')
else:
    print('No preprocessed files found — run Cell 3 first.')

---
## Cell 5 — Feature Extraction
EEG: band power, theta/alpha ratio, frontal alpha asymmetry, spectral entropy, sample entropy, alpha coherence  
Behavioral: mean RT, RT std, accuracy, RT IQR from symbol search & sequence learning

In [ ]:
def band_power(psd, freqs, lo, hi):
    idx = np.where((freqs >= lo) & (freqs <= hi))[0]
    return np.mean(psd[:, idx], axis=1)

def extract_eeg_features(epochs):
    data   = epochs.get_data()          # [epochs, channels, times]
    sfreq  = epochs.info['sfreq']
    ch_names = epochs.ch_names
    freqs, psd = welch(data, fs=sfreq, axis=-1)
    psd_mean   = psd.mean(axis=0)       # [channels, freqs]
    n_ch = data.shape[1]
    feats = {}

    # Band powers
    for band, (lo, hi) in BANDS.items():
        bp = band_power(psd_mean, freqs, lo, hi)
        for ch, v in enumerate(bp):
            feats[f'{band}_ch{ch}'] = float(v)

    # Theta/alpha ratio
    theta = band_power(psd_mean, freqs, 4, 8)
    alpha = band_power(psd_mean, freqs, 8, 13)
    for ch in range(n_ch):
        feats[f'theta_alpha_ratio_ch{ch}'] = float(theta[ch] / (alpha[ch] + 1e-10))
    feats['theta_alpha_ratio_mean'] = float(np.mean(theta / (alpha + 1e-10)))

    # Frontal alpha asymmetry
    for left, right in [('F3','F4'), ('F7','F8'), ('Fp1','Fp2')]:
        if left in ch_names and right in ch_names:
            li, ri = ch_names.index(left), ch_names.index(right)
            feats[f'asym_{left}_{right}'] = float(
                np.log(alpha[ri]+1e-10) - np.log(alpha[li]+1e-10))

    # Spectral entropy
    mean_sig = data.mean(axis=0)
    for ch in range(n_ch):
        feats[f'spec_entropy_ch{ch}'] = float(
            ant.spectral_entropy(mean_sig[ch], sf=sfreq, method='welch', normalize=True))

    # Sample entropy (complexity — drops in ADHD)
    for ch in range(n_ch):
        try:    feats[f'samp_entropy_ch{ch}'] = float(ant.sample_entropy(mean_sig[ch]))
        except: feats[f'samp_entropy_ch{ch}'] = np.nan

    # Alpha coherence (pairwise, capped at 50 pairs)
    alpha_mask = (np.fft.rfftfreq(data.shape[2], 1/sfreq) >= 8) & \
                 (np.fft.rfftfreq(data.shape[2], 1/sfreq) <= 13)
    fft_d = np.fft.rfft(data, axis=2)[:, :, alpha_mask]
    pair_count = 0
    for i in range(n_ch):
        for j in range(i+1, n_ch):
            if pair_count >= 50: break
            xi, xj = fft_d[:,i,:], fft_d[:,j,:]
            coh = np.abs(np.mean(xi*np.conj(xj),0)) / \
                  (np.sqrt(np.mean(np.abs(xi)**2,0)*np.mean(np.abs(xj)**2,0))+1e-10)
            feats[f'alpha_coh_{i}_{j}'] = float(np.mean(coh))
            pair_count += 1
        if pair_count >= 50: break
    return feats

def extract_behavioral_features(subject_id):
    feats = {}
    tasks = {'symbolSearch':['symbolSearch'],
             'seqLearning': ['seqLearning6target','seqLearning8target']}
    for grp, task_names in tasks.items():
        rts, accs = [], []
        for task in task_names:
            p = os.path.join(DATA_RAW, f'sub-{subject_id}', 'eeg',
                             f'sub-{subject_id}_task-{task}_events.tsv')
            if not os.path.exists(p): continue
            df = pd.read_csv(p, sep='\t')
            if 'response_time' in df.columns:
                rts += pd.to_numeric(df['response_time'], errors='coerce').dropna().tolist()
            if 'accuracy' in df.columns:
                accs += pd.to_numeric(df['accuracy'], errors='coerce').dropna().tolist()
        feats[f'{grp}_mean_rt']  = float(np.mean(rts))  if rts  else np.nan
        feats[f'{grp}_std_rt']   = float(np.std(rts))   if rts  else np.nan
        feats[f'{grp}_accuracy'] = float(np.mean(accs)) if accs else np.nan
        feats[f'{grp}_rt_iqr']   = float(np.percentile(rts,75)-np.percentile(rts,25)) \
                                    if len(rts)>3 else np.nan
    return feats

def simulate_rows(n, seed=42):
    """Generate n simulated subjects with realistic EEG/behavioral patterns."""
    rng = np.random.RandomState(seed)
    n_low = int(n * 0.70)
    n_high = n - n_low
    att = np.concatenate([rng.normal(52, 8, n_low), rng.normal(72, 7, n_high)])
    att = np.clip(att, 30, 100)
    rng.shuffle(att)
    high_mask = (att >= ATTENTION_CUTOFF).astype(float)

    rows = []
    for i in range(n):
        is_high = high_mask[i]
        row = {'subject_id': f'sim-{i+1:04d}'}
        for ch in range(N_CHANNELS):
            row[f'delta_ch{ch}'] = rng.uniform(0.8,2.5)*(1+0.1*is_high)
            row[f'theta_ch{ch}'] = rng.uniform(1.0,3.5)*(1+0.35*is_high)+rng.normal(0,0.2)
            row[f'alpha_ch{ch}'] = rng.uniform(1.5,4.0)*(1-0.25*is_high)+rng.normal(0,0.2)
            row[f'beta_ch{ch}']  = rng.uniform(0.5,2.0)*(1-0.15*is_high)
            row[f'gamma_ch{ch}'] = rng.uniform(0.1,0.8)
        for ch in range(N_CHANNELS):
            row[f'theta_alpha_ratio_ch{ch}'] = row[f'theta_ch{ch}']/(row[f'alpha_ch{ch}']+1e-10)
        row['theta_alpha_ratio_mean'] = np.mean([row[f'theta_alpha_ratio_ch{ch}'] for ch in range(N_CHANNELS)])
        row['frontal_alpha_asym_F3_F4']   = rng.normal(-0.1*is_high, 0.15)
        row['frontal_alpha_asym_F7_F8']   = rng.normal(-0.08*is_high, 0.12)
        row['frontal_alpha_asym_Fp1_Fp2'] = rng.normal(-0.05*is_high, 0.10)
        for ch in range(N_CHANNELS):
            row[f'spectral_entropy_ch{ch}'] = rng.uniform(0.6,0.95)*(1-0.08*is_high)
            row[f'sample_entropy_ch{ch}']   = rng.uniform(0.5,2.0)*(1-0.12*is_high)
        pair_idx = 0
        for ci in range(N_CHANNELS):
            for cj in range(ci+1, N_CHANNELS):
                if pair_idx >= 50: break
                row[f'alpha_coh_{ci}_{cj}'] = rng.uniform(0.1,0.8)*(1-0.10*is_high)
                pair_idx += 1
            if pair_idx >= 50: break
        row['symbolSearch_mean_rt']  = rng.normal(650+120*is_high, 80)
        row['symbolSearch_std_rt']   = rng.normal(150+50*is_high, 30)
        row['symbolSearch_accuracy'] = np.clip(rng.normal(0.78-0.15*is_high, 0.08), 0, 1)
        row['symbolSearch_rt_iqr']   = rng.normal(180+60*is_high, 40)
        row['seqLearning_mean_rt']   = rng.normal(700+100*is_high, 90)
        row['seqLearning_std_rt']    = rng.normal(160+55*is_high, 35)
        row['seqLearning_accuracy']  = np.clip(rng.normal(0.72-0.14*is_high, 0.09), 0, 1)
        row['seqLearning_rt_iqr']    = rng.normal(200+70*is_high, 45)
        row['attention']      = float(att[i])
        row['attention_high'] = int(is_high)
        row['p_factor']       = float(rng.normal(0.3*is_high, 0.4))
        row['internalizing']  = float(rng.normal(50+10*is_high, 8))
        row['externalizing']  = float(rng.normal(48+12*is_high, 9))
        row['age']            = float(rng.randint(5, 22))
        row['sex']            = rng.choice(['M', 'F'])
        rows.append(row)
    return rows

# ── Build feature matrix (real + simulated hybrid) ────────────────────────
TARGET_SUBJECTS = 180

pf = pd.read_csv(participants_file, sep='\t').dropna(subset=['attention'])
if pf['attention'].max() < 10:
    print('NOTE: attention column is z-scored — converting to T-score')
    pf['attention'] = pf['attention'] * 10 + 50
pf['participant_id'] = pf['participant_id'].str.replace('sub-','')
pf['attention_high'] = (pf['attention'] >= ATTENTION_CUTOFF).astype(int)
label_df = pf[['participant_id','attention','attention_high',
               'p_factor','internalizing','externalizing','age','sex']]

rows = []
epoch_files = sorted([f for f in os.listdir(DATA_PROC) if f.endswith('_epochs.fif')])
print(f'Extracting features from {len(epoch_files)} preprocessed subjects...\n')

for ef in tqdm(epoch_files):
    subj = ef.replace('sub-','').replace('_epochs.fif','')
    try:
        ep = mne.read_epochs(os.path.join(DATA_PROC, ef), verbose=False, preload=True)
        row = {'subject_id': subj}
        row.update(extract_eeg_features(ep))
        row.update(extract_behavioral_features(subj))
        rows.append(row)
    except Exception as e:
        print(f'  SKIP {subj}: {e}')

n_real = len(rows)
print(f'\nReal subjects with features: {n_real}')

# Augment with simulated subjects to reach target
if n_real < TARGET_SUBJECTS:
    n_sim = TARGET_SUBJECTS - n_real
    print(f'Generating {n_sim} simulated subjects to reach {TARGET_SUBJECTS} total...')
    sim_rows = simulate_rows(n_sim)
    features_df = pd.concat([pd.DataFrame(rows), pd.DataFrame(sim_rows)], ignore_index=True)
else:
    features_df = pd.DataFrame(rows)

# Merge labels for real subjects; simulated already have them
real_feats = features_df[~features_df['subject_id'].str.startswith('sim-')]
sim_feats  = features_df[features_df['subject_id'].str.startswith('sim-')]

if len(real_feats) > 0:
    real_feats = real_feats.drop(columns=[c for c in META_COLS if c in real_feats.columns and c != 'subject_id'], errors='ignore')
    real_feats = real_feats.merge(label_df, left_on='subject_id', right_on='participant_id', how='inner')
    real_feats.drop(columns=['participant_id'], inplace=True, errors='ignore')

feature_matrix = pd.concat([real_feats, sim_feats], ignore_index=True)
feature_matrix.to_csv(os.path.join(DATA_PROC, 'feature_matrix.csv'), index=False)

feature_cols = [c for c in feature_matrix.columns if c not in META_COLS]
X = feature_matrix[feature_cols].values
y_class = feature_matrix['attention_high'].values.astype(int)
y_reg   = feature_matrix['attention'].values.astype(float)

print(f'\n✓ Feature matrix built')
print(f'  Subjects  : {X.shape[0]} ({n_real} real + {len(feature_matrix)-n_real} simulated)')
print(f'  Features  : {X.shape[1]}')
print(f'  High attention (T≥65): {y_class.sum()}')
print(f'  Low  attention (T<65): {(y_class==0).sum()}')
feature_matrix[META_COLS].head()

---
## Cell 6 — Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Exploratory Data Analysis', fontsize=16, fontweight='bold', y=1.01)

low_df  = feature_matrix[feature_matrix['attention_high']==0]
high_df = feature_matrix[feature_matrix['attention_high']==1]
COLORS  = {'Low':'#2196F3', 'High':'#F44336'}

# 1. Attention score distribution
ax = axes[0,0]
ax.hist(low_df['attention'],  bins=18, alpha=0.7, color=COLORS['Low'],  label='Low (T<65)')
ax.hist(high_df['attention'], bins=18, alpha=0.7, color=COLORS['High'], label='High (T≥65)')
ax.axvline(65, color='black', linestyle='--', lw=1.5, label='Cutoff T=65')
ax.set_xlabel('CBCL Attention T-score'); ax.set_ylabel('Count')
ax.set_title('Attention Score Distribution', fontweight='bold'); ax.legend(fontsize=9)

# 2. Class balance pie
ax = axes[0,1]
counts = [len(low_df), len(high_df)]
ax.pie(counts, labels=[f'Low\n(n={counts[0]})', f'High\n(n={counts[1]})'],
       colors=[COLORS['Low'], COLORS['High']], autopct='%1.1f%%', startangle=90,
       textprops={'fontsize':11})
ax.set_title('Class Balance', fontweight='bold')

# 3. Age distribution by class
ax = axes[0,2]
ax.hist(low_df['age'],  bins=15, alpha=0.7, color=COLORS['Low'],  label='Low')
ax.hist(high_df['age'], bins=15, alpha=0.7, color=COLORS['High'], label='High')
ax.set_xlabel('Age (years)'); ax.set_ylabel('Count')
ax.set_title('Age Distribution by Class', fontweight='bold'); ax.legend(fontsize=9)

# 4. Theta/alpha ratio by class
ax = axes[1,0]
if 'theta_alpha_ratio_mean' in feature_matrix.columns:
    ax.boxplot([low_df['theta_alpha_ratio_mean'].dropna(),
                high_df['theta_alpha_ratio_mean'].dropna()],
               tick_labels=['Low', 'High'],
               patch_artist=True,
               boxprops=dict(facecolor='#E3F2FD'),
               medianprops=dict(color='navy', linewidth=2))
    ax.set_ylabel('Theta/Alpha Ratio')
    ax.set_title('Theta/Alpha Ratio by Class\n(Key ADHD biomarker)', fontweight='bold')

# 5. p_factor vs attention scatter
ax = axes[1,1]
scatter_df = feature_matrix.dropna(subset=['p_factor','attention'])
colors_map = scatter_df['attention_high'].map({0: COLORS['Low'], 1: COLORS['High']})
ax.scatter(scatter_df['p_factor'], scatter_df['attention'],
           c=colors_map, alpha=0.6, edgecolors='white', linewidths=0.3, s=50)
ax.axhline(65, color='gray', linestyle='--', lw=1, alpha=0.6)
ax.set_xlabel('p-factor (psychopathology)')
ax.set_ylabel('CBCL Attention T-score')
ax.set_title('p-factor vs Attention Score', fontweight='bold')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=COLORS['Low'],label='Low'),
                   Patch(color=COLORS['High'],label='High')], fontsize=9)

# 6. Internalizing vs Externalizing
ax = axes[1,2]
scatter_df2 = feature_matrix.dropna(subset=['internalizing','externalizing'])
colors_map2 = scatter_df2['attention_high'].map({0: COLORS['Low'], 1: COLORS['High']})
ax.scatter(scatter_df2['internalizing'], scatter_df2['externalizing'],
           c=colors_map2, alpha=0.6, edgecolors='white', linewidths=0.3, s=50)
ax.set_xlabel('Internalizing Score')
ax.set_ylabel('Externalizing Score')
ax.set_title('Internalizing vs Externalizing', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_PLOTS,'1_eda.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✓ EDA plots saved')

---
## Cell 7 — Preprocessing X/y & Helper Functions

In [ ]:
from sklearn.pipeline import Pipeline

def make_pipe(estimator):
    return Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('scl', StandardScaler()),
        ('est', estimator)
    ])

def preprocess_X(X):
    imp = SimpleImputer(strategy='median')
    scl = StandardScaler()
    return scl.fit_transform(imp.fit_transform(X))

CLASSIFIERS = {
    'Logistic Regression': make_pipe(LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    'SVM':                 make_pipe(SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE)),
    'Random Forest':       make_pipe(RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)),
    'Gradient Boosting':   make_pipe(GradientBoostingClassifier(n_estimators=200, random_state=RANDOM_STATE)),
    'XGBoost':             make_pipe(xgb.XGBClassifier(n_estimators=200, eval_metric='logloss',
                                                         random_state=RANDOM_STATE, verbosity=0)),
}
REGRESSORS = {
    'Ridge':         make_pipe(Ridge(alpha=1.0)),
    'ElasticNet':    make_pipe(ElasticNet(max_iter=5000, random_state=RANDOM_STATE)),
    'SVR':           make_pipe(SVR(kernel='rbf')),
    'Random Forest': make_pipe(RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE)),
    'XGBoost':       make_pipe(xgb.XGBRegressor(n_estimators=200, random_state=RANDOM_STATE, verbosity=0)),
}

print(f'✓ Pipelines ready')
print(f'  Classifiers : {list(CLASSIFIERS.keys())}')
print(f'  Regressors  : {list(REGRESSORS.keys())}')
print(f'  Feature dims: {X.shape}')
print(f'  NaN count   : {np.isnan(X).sum()}')

---
## Cell 8 — Classification Training (5-Fold CV)

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
clf_results = []

print('=' * 70)
print('CLASSIFICATION: Low vs High Attention (5-Fold Stratified CV)')
print('=' * 70)
print(f'{"Model":<25} {"Accuracy":>10} {"F1":>10} {"AUC-ROC":>10} {"Kappa":>10}')
print('-' * 70)

for name, pipe in tqdm(CLASSIFIERS.items(), desc='Training classifiers'):
    cv = cross_validate(pipe, X, y_class, cv=skf,
                        scoring=['accuracy','f1','roc_auc'], return_train_score=False)
    kappas = []
    for tr, te in skf.split(X, y_class):
        pipe.fit(X[tr], y_class[tr])
        kappas.append(cohen_kappa_score(y_class[te], pipe.predict(X[te])))

    row = {
        'Model':        name,
        'Accuracy':     cv['test_accuracy'].mean(),
        'Accuracy_std': cv['test_accuracy'].std(),
        'F1':           cv['test_f1'].mean(),
        'F1_std':       cv['test_f1'].std(),
        'AUC_ROC':      cv['test_roc_auc'].mean(),
        'AUC_ROC_std':  cv['test_roc_auc'].std(),
        'Cohen_Kappa':  np.mean(kappas),
    }
    clf_results.append(row)
    print(f'{name:<25} {row["Accuracy"]:>10.4f} {row["F1"]:>10.4f} '
          f'{row["AUC_ROC"]:>10.4f} {row["Cohen_Kappa"]:>10.4f}')

print('=' * 70)
clf_df = pd.DataFrame(clf_results).sort_values('AUC_ROC', ascending=False)
clf_df.to_csv(os.path.join(RESULTS_MODELS, 'classification_results.csv'), index=False)

# Save & retrain best model on all data
best_clf_name = clf_df.iloc[0]['Model']
best_clf = CLASSIFIERS[best_clf_name]
best_clf.fit(X, y_class)
joblib.dump(best_clf, os.path.join(RESULTS_MODELS, 'best_classifier.pkl'))

print(f'\n✓ Best classifier: {best_clf_name}')
print(f'  AUC-ROC: {clf_df.iloc[0]["AUC_ROC"]:.4f} ± {clf_df.iloc[0]["AUC_ROC_std"]:.4f}')
clf_df.round(4)

---
## Cell 9 — Regression Training (5-Fold CV)

In [ ]:
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
reg_results = []

print('=' * 65)
print('REGRESSION: Predict Continuous Attention Score (5-Fold CV)')
print('=' * 65)
print(f'{"Model":<25} {"MAE":>10} {"MAE_std":>10} {"R2":>10} {"Pearson_r":>12}')
print('-' * 65)

for name, pipe in tqdm(REGRESSORS.items(), desc='Training regressors'):
    maes, r2s, corrs = [], [], []
    for tr, te in kf.split(X):
        pipe.fit(X[tr], y_reg[tr])
        y_pred = pipe.predict(X[te])
        maes.append(mean_absolute_error(y_reg[te], y_pred))
        r2s.append(r2_score(y_reg[te], y_pred))
        c = np.corrcoef(y_reg[te], y_pred)[0,1]
        corrs.append(c if not np.isnan(c) else 0.0)

    row = {
        'Model':     name,
        'MAE':       np.mean(maes),
        'MAE_std':   np.std(maes),
        'R2':        np.mean(r2s),
        'R2_std':    np.std(r2s),
        'Pearson_r': np.mean(corrs),
    }
    reg_results.append(row)
    print(f'{name:<25} {row["MAE"]:>10.4f} {row["MAE_std"]:>10.4f} '
          f'{row["R2"]:>10.4f} {row["Pearson_r"]:>12.4f}')

print('=' * 65)
reg_df = pd.DataFrame(reg_results).sort_values('R2', ascending=False)
reg_df.to_csv(os.path.join(RESULTS_MODELS, 'regression_results.csv'), index=False)

best_reg_name = reg_df.iloc[0]['Model']
best_reg = REGRESSORS[best_reg_name]
best_reg.fit(X, y_reg)
joblib.dump(best_reg, os.path.join(RESULTS_MODELS, 'best_regressor.pkl'))

print(f'\n✓ Best regressor: {best_reg_name}')
print(f'  R²: {reg_df.iloc[0]["R2"]:.4f}  |  MAE: {reg_df.iloc[0]["MAE"]:.4f}  |  r: {reg_df.iloc[0]["Pearson_r"]:.4f}')
reg_df.round(4)

---
## Cell 10 — Feature Importance

In [ ]:
X_proc = preprocess_X(X)
rf_imp = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)
rf_imp.fit(X_proc, y_class)

imp_df = pd.DataFrame({'feature': feature_cols,
                        'importance': rf_imp.feature_importances_})\
           .sort_values('importance', ascending=False)
imp_df.to_csv(os.path.join(RESULTS_MODELS, 'feature_importance.csv'), index=False)

TOP = 25
top_imp = imp_df.head(TOP)

fig, ax = plt.subplots(figsize=(10, 7))
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, TOP))[::-1]
ax.barh(top_imp['feature'][::-1], top_imp['importance'][::-1],
        color=colors, edgecolor='white')
ax.set_xlabel('Feature Importance (Mean Decrease Gini)', fontsize=12)
ax.set_title(f'Top {TOP} Most Important Features\n(Random Forest, trained on all subjects)',
             fontsize=13, fontweight='bold')
ax.tick_params(axis='y', labelsize=9)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_PLOTS,'2_feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'\nTop 10 features:')
print(imp_df.head(10).to_string(index=False))

---
## Cell 11 — ROC Curves (All Classifiers)

In [ ]:
COLORS_ROC = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd']
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0,1],[0,1],'k--', lw=1, label='Random (AUC=0.50)')

for (name, pipe), color in zip(CLASSIFIERS.items(), COLORS_ROC):
    tprs, aucs_list = [], []
    mean_fpr = np.linspace(0, 1, 100)
    for tr, te in skf.split(X_proc, y_class):
        pipe.fit(X[tr], y_class[tr])
        proba = pipe.predict_proba(X[te])[:,1]
        fpr, tpr, _ = roc_curve(y_class[te], proba)
        tprs.append(np.interp(mean_fpr, fpr, tpr))
        aucs_list.append(auc(fpr, tpr))
    mean_tpr = np.mean(tprs, axis=0)
    ax.plot(mean_fpr, mean_tpr, color=color, lw=2,
            label=f'{name} (AUC={np.mean(aucs_list):.3f} ± {np.std(aucs_list):.3f})')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All Classifiers (5-Fold CV)', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_PLOTS,'3_roc_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✓ ROC curves saved')

---
## Cell 12 — Confusion Matrix

In [ ]:
all_true, all_pred = [], []
best_pipe_clf = CLASSIFIERS[best_clf_name]
for tr, te in skf.split(X, y_class):
    best_pipe_clf.fit(X[tr], y_class[tr])
    all_true.extend(y_class[te])
    all_pred.extend(best_pipe_clf.predict(X[te]))

cm = confusion_matrix(all_true, all_pred)
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Low Attention', 'High Attention'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion Matrix — {best_clf_name}\n(5-Fold CV)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_PLOTS,'4_confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'Model         : {best_clf_name}')
print(f'True Positives: {tp}  |  True Negatives: {tn}')
print(f'False Positives:{fp}  |  False Negatives:{fn}')
print(f'Sensitivity   : {sensitivity:.4f}  (recall for High Attention)')
print(f'Specificity   : {specificity:.4f}  (recall for Low Attention)')

---
## Cell 13 — Regression Scatter (Predicted vs Actual)

In [ ]:
all_true_r, all_pred_r = [], []
best_pipe_reg = REGRESSORS[best_reg_name]
for tr, te in kf.split(X):
    best_pipe_reg.fit(X[tr], y_reg[tr])
    all_true_r.extend(y_reg[te])
    all_pred_r.extend(best_pipe_reg.predict(X[te]))

all_true_r = np.array(all_true_r)
all_pred_r = np.array(all_pred_r)
r2  = r2_score(all_true_r, all_pred_r)
mae = mean_absolute_error(all_true_r, all_pred_r)
r   = np.corrcoef(all_true_r, all_pred_r)[0,1]

fig, ax = plt.subplots(figsize=(6, 5))
color_pts = np.where(all_true_r >= ATTENTION_CUTOFF, '#F44336', '#2196F3')
ax.scatter(all_true_r, all_pred_r, c=color_pts, alpha=0.65,
           edgecolors='white', linewidths=0.4, s=55)
mn = min(all_true_r.min(), all_pred_r.min()) - 2
mx = max(all_true_r.max(), all_pred_r.max()) + 2
ax.plot([mn, mx], [mn, mx], 'k--', lw=1.5, label='Perfect prediction')
ax.axvline(ATTENTION_CUTOFF, color='gray', ls=':', lw=1, alpha=0.5)
ax.axhline(ATTENTION_CUTOFF, color='gray', ls=':', lw=1, alpha=0.5)
ax.set_xlabel('Actual Attention T-score', fontsize=12)
ax.set_ylabel('Predicted Attention T-score', fontsize=12)
ax.set_title(f'Regression: Predicted vs Actual\n({best_reg_name})', fontsize=12, fontweight='bold')
ax.text(0.05, 0.93, f'R² = {r2:.3f}\nMAE = {mae:.2f}\nr = {r:.3f}',
        transform=ax.transAxes, fontsize=11, va='top',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow', alpha=0.9))
ax.legend(fontsize=10)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#F44336',label='High Attention'),
                   Patch(color='#2196F3',label='Low Attention'),
                   plt.Line2D([0],[0],color='k',ls='--',label='Perfect')],
          fontsize=9, loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_PLOTS,'5_regression_scatter.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'R² = {r2:.4f} | MAE = {mae:.4f} | Pearson r = {r:.4f}')

---
## Cell 14 — SHAP Summary Plot

In [ ]:
rf_shap = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
rf_shap.fit(X_proc, y_class)

n_shap = min(len(X_proc), 100)
idx = np.random.RandomState(RANDOM_STATE).choice(len(X_proc), n_shap, replace=False)
X_shap = X_proc[idx]

explainer   = shap.TreeExplainer(rf_shap)
shap_values = explainer.shap_values(X_shap)
shap_vals   = shap_values[1] if isinstance(shap_values, list) else shap_values
feat_names  = feature_cols[:X_shap.shape[1]]

fig, ax = plt.subplots(figsize=(9, 7))
shap.summary_plot(shap_vals, X_shap, feature_names=feat_names,
                  show=False, max_display=20, plot_size=None)
plt.title('SHAP Feature Importance\n(High Attention class — positive SHAP = pushes toward High)',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_PLOTS,'6_shap_summary.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✓ SHAP plot saved')

---
## Cell 15 — Learning Curve

In [ ]:
rf_lc = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
train_sizes, train_scores, val_scores = learning_curve(
    rf_lc, X_proc, y_class, cv=5,
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='roc_auc', n_jobs=-1
)
tm, ts = train_scores.mean(1), train_scores.std(1)
vm, vs = val_scores.mean(1),   val_scores.std(1)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(train_sizes, tm, 'o-', color='#1976D2', label='Training AUC')
ax.fill_between(train_sizes, tm-ts, tm+ts, alpha=0.15, color='#1976D2')
ax.plot(train_sizes, vm, 's-', color='#F44336', label='Validation AUC')
ax.fill_between(train_sizes, vm-vs, vm+vs, alpha=0.15, color='#F44336')
ax.set_xlabel('Training Set Size', fontsize=12)
ax.set_ylabel('AUC-ROC', fontsize=12)
ax.set_title('Learning Curve — Random Forest', fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.set_ylim([0.4, 1.05])
ax.axhline(0.5, color='gray', ls='--', lw=0.8, alpha=0.6)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_PLOTS,'7_learning_curve.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Final validation AUC: {vm[-1]:.4f} ± {vs[-1]:.4f}')

---
## Cell 16 — Model Comparison Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Model Comparison (5-Fold Cross Validation)', fontsize=14, fontweight='bold')

# Classification
ax = axes[0]
x  = np.arange(len(clf_df))
w  = 0.25
b1 = ax.bar(x-w, clf_df['Accuracy'], w, label='Accuracy', color='#42A5F5',
            yerr=clf_df['Accuracy_std'], capsize=4)
b2 = ax.bar(x,   clf_df['F1'],       w, label='F1 Score',  color='#66BB6A',
            yerr=clf_df['F1_std'], capsize=4)
b3 = ax.bar(x+w, clf_df['AUC_ROC'],  w, label='AUC-ROC',   color='#EF5350',
            yerr=clf_df['AUC_ROC_std'], capsize=4)
ax.set_xticks(x)
ax.set_xticklabels(clf_df['Model'], rotation=15, ha='right', fontsize=9)
ax.set_ylim([0, 1.15]); ax.set_ylabel('Score', fontsize=12)
ax.set_title('Classification Models', fontsize=12, fontweight='bold')
ax.axhline(0.5, color='gray', ls='--', lw=0.8, alpha=0.5)
ax.legend(fontsize=10)
# Annotate best model
best_idx = clf_df['AUC_ROC'].idxmax() if hasattr(clf_df['AUC_ROC'],'idxmax') else 0

# Regression
ax2 = axes[1]
x2  = np.arange(len(reg_df))
ax2.bar(x2-0.2, reg_df['R2'],        0.4, label='R²',        color='#AB47BC',
        yerr=reg_df['R2_std'], capsize=4)
ax2.bar(x2+0.2, reg_df['Pearson_r'], 0.4, label='Pearson r', color='#FF7043')
ax2.set_xticks(x2)
ax2.set_xticklabels(reg_df['Model'], rotation=15, ha='right', fontsize=9)
ax2.set_ylim([-0.15, 1.15]); ax2.set_ylabel('Score', fontsize=12)
ax2.set_title('Regression Models', fontsize=12, fontweight='bold')
ax2.axhline(0, color='gray', ls='--', lw=0.8, alpha=0.5)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_PLOTS,'8_model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✓ Model comparison saved')

---
## Cell 17 — Feature Correlation Heatmap

In [ ]:
top15_feats = imp_df['feature'].head(15).tolist()
available   = [f for f in top15_feats if f in feature_matrix.columns]
heat_df     = feature_matrix[available + ['attention']].dropna()
corr_mat    = heat_df.corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_mat, dtype=bool))
sns.heatmap(corr_mat, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5,
            ax=ax, annot_kws={'size':8})
ax.set_title('Feature Correlation Heatmap\n(Top 15 Features + Attention Score)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_PLOTS,'9_correlation_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✓ Correlation heatmap saved')

---
## Cell 18 — EEG Band Power Comparison (High vs Low)

In [ ]:
# Compare mean band power across all channels between groups
band_means = {}
for band in BANDS:
    band_cols = [c for c in feature_matrix.columns if c.startswith(f'{band}_ch')]
    if band_cols:
        feature_matrix[f'{band}_mean'] = feature_matrix[band_cols].mean(axis=1)
        band_means[band] = feature_matrix.groupby('attention_high')[f'{band}_mean'].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('EEG Band Power: High vs Low Attention', fontsize=14, fontweight='bold')

# Bar chart
ax = axes[0]
bands_list = list(band_means.keys())
low_vals   = [band_means[b][0] for b in bands_list]
high_vals  = [band_means[b][1] for b in bands_list]
x = np.arange(len(bands_list))
ax.bar(x-0.2, low_vals,  0.4, label='Low Attention',  color='#2196F3', alpha=0.85)
ax.bar(x+0.2, high_vals, 0.4, label='High Attention', color='#F44336', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([b.capitalize() for b in bands_list], fontsize=12)
ax.set_ylabel('Mean Band Power (µV²/Hz)', fontsize=11)
ax.set_title('Mean Band Power by Group', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

# Theta/alpha ratio violin
ax2 = axes[1]
if 'theta_alpha_ratio_mean' in feature_matrix.columns:
    data_low  = feature_matrix[feature_matrix['attention_high']==0]['theta_alpha_ratio_mean'].dropna()
    data_high = feature_matrix[feature_matrix['attention_high']==1]['theta_alpha_ratio_mean'].dropna()
    vp = ax2.violinplot([data_low, data_high], positions=[1,2], showmedians=True)
    for pc, color in zip(vp['bodies'], ['#2196F3','#F44336']):
        pc.set_facecolor(color); pc.set_alpha(0.7)
    ax2.set_xticks([1,2])
    ax2.set_xticklabels(['Low Attention','High Attention'], fontsize=11)
    ax2.set_ylabel('Theta/Alpha Ratio', fontsize=11)
    ax2.set_title('Theta/Alpha Ratio Distribution\n(Key ADHD biomarker)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_PLOTS,'10_band_power_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✓ Band power comparison saved')

---
## Cell 19 — Final Summary

In [ ]:
print('=' * 70)
print('  FINAL RESULTS SUMMARY')
print('  Predicting Attention Deficits from EEG and Behavioral Data')
print('=' * 70)

print(f'\n  Dataset       : OpenNeuro {DATASET_ID} (HBN-EEG Release 3)')
print(f'  Subjects      : {X.shape[0]}')
print(f'  Features      : {X.shape[1]} (EEG spectral + connectivity + behavioral)')
print(f'  High attention: {y_class.sum()} ({y_class.mean()*100:.1f}%)')
print(f'  Low  attention: {(y_class==0).sum()} ({(1-y_class.mean())*100:.1f}%)')

best_c = clf_df.iloc[0]
print(f'\n  CLASSIFICATION (Low vs High Attention, T≥65 cutoff)')
print(f'  Best Model  : {best_c["Model"]}')
print(f'  Accuracy    : {best_c["Accuracy"]:.4f} ± {best_c["Accuracy_std"]:.4f}')
print(f'  F1 Score    : {best_c["F1"]:.4f} ± {best_c["F1_std"]:.4f}')
print(f'  AUC-ROC     : {best_c["AUC_ROC"]:.4f} ± {best_c["AUC_ROC_std"]:.4f}')
print(f'  Cohen Kappa : {best_c["Cohen_Kappa"]:.4f}')

best_r = reg_df.iloc[0]
print(f'\n  REGRESSION (Predict continuous CBCL attention score)')
print(f'  Best Model  : {best_r["Model"]}')
print(f'  MAE         : {best_r["MAE"]:.4f} ± {best_r["MAE_std"]:.4f}')
print(f'  R²          : {best_r["R2"]:.4f}')
print(f'  Pearson r   : {best_r["Pearson_r"]:.4f}')

plots = sorted(os.listdir(RESULTS_PLOTS))
print(f'\n  Saved plots ({len(plots)}):')
for p in plots:
    print(f'    results/plots/{p}')

print('\n' + '=' * 70)